### Testing Hugging Face Transformers

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

c:\Users\rzamb\Documents\llm-inference-stack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [12]:
input_text = "Once upon a time there was a dog who would speak english instead of barking"
inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=100,
    do_sample=True,         # critical
    temperature=0.9,        # randomness
    top_k=50,               # limit choices
    top_p=0.95,             # nucleus sampling
    repetition_penalty=1.2, # discourages loops
    )
decoded_output = tokenizer.decode(outputs[0])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [13]:
print(decoded_output)

Once upon a time there was a dog who would speak english instead of barking. I remember seeing his wife, her children (whom he had met on the street), one more person to talk about their child's name and they were sitting up talking it out in front from where she sat beside him as if having something that sounded like hers or said 'I'm sorry…'. Then again this particular puppy came over next week with little Aussie accent just because these people all talked English well enough for


### Testing ONNX serialized model

In [14]:
from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

In [18]:
tokenizer = AutoTokenizer.from_pretrained("model_onnx")
#tokenizer.pad_token = tokenizer.eos_token

onnx_model = ORTModelForCausalLM.from_pretrained(
    "model_onnx",
    #export=True,
    #use_cache=True,
    )

In [ ]:
inputs = tokenizer("Once upon a time there was a dog dancing merengue", return_tensors="pt")
outputs = onnx_model.generate(
    **inputs, 
    max_length=100,
    do_sample=True,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
    )

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [24]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Once upon a time there was a dog dancing merengue and no one in the room could see it. There were some men on both sides of the table, who gave way to others as they stared at him; but when all seemed lost he went out with his back turned towards them without looking up from their hands or feet until what had been an empty space filled by people sat down quietly behind each other (see below).
They continued for another hour more than twelve hours—two long periods


### Testing quantized model

In [25]:
# Load and run
tokenizer = AutoTokenizer.from_pretrained("model_onnx_quantized")
quantized_model = ORTModelForCausalLM.from_pretrained("model_onnx_quantized")

In [ ]:
inputs = tokenizer("Once upon a time there was a dog that was able to memorize entire books", return_tensors="pt")
outputs = quantized_model.generate(
    **inputs, 
    max_length=100,
    do_sample=True,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
    )

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [29]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Once upon a time there was a dog that was able to memorize entire books in 20 minutes, and it did so by walking through an entirely new building. It is known as the "Duckman of Chicago."
)


### Difference between serving platforms:
<br>
* FastAPI:   Python code IS the server   > you write endpoints <br>
* BentoML:   Python class IS the service > BentoML generates the server <br>
* Triton:    config file IS the server   > no Python needed for inference, only for pre/post processing <br>
* vLLM:      [Linux first, skipped :(]
